In [1]:
import numpy as np
from collections import Counter
import random
import pandas as pd

In [5]:
combinations = {
    (1,): 100, (5,): 50, 
    3 * (1,): 1000, 3 * (2,): 200, 3 * (3,): 300, 3 * (4,): 400, 3 * (5,): 500, 3 * (6,): 600,
    4 * (1,): 2*1000, 4 * (2,): 2*200, 4 * (3,): 2*300, 4 * (4,): 2*400, 4 * (5,): 2*500, 4 * (6,): 2*600,
    5 * (1,): 4*1000, 5 * (2,): 4*200, 5 * (3,): 4*300, 5 * (4,): 4*400, 5 * (5,): 4*500, 5 * (6,): 4*600,
    6 * (1,): 8*1000, 6 * (2,): 8*200, 6 * (3,): 8*300, 6 * (4,): 8*400, 6 * (5,): 8*500, 6 * (6,): 8*600,
    (1, 2, 3, 4, 5, 6): 1500, (1, 2, 3, 4, 5): 500, (2, 3, 4, 5, 6): 750
}

In [6]:
def find_scores(dice_counts):
    results = []

    def backtrack(current_counts, current_path, current_score, used_count):
        # Если в пути что-то есть, сохраняем это как вариант
        if current_path:
            results.append((current_path, current_score, 6 - used_count))
        
        # Перебираем все возможные комбинации из словаря
        for combo, points in combinations.items():
            combo_counts = Counter(combo)
            # Проверяем, можно ли взять эту комбинацию
            if all(current_counts[val] >= count for val, count in combo_counts.items()):
                # Уменьшаем счетчик костей
                new_counts = current_counts - combo_counts
                # Рекурсивно ищем дальше, передавая отсортированный путь для избежания повторов
                new_path = tuple(sorted(current_path + (combo,)))
                backtrack(new_counts, new_path, current_score + points, used_count + len(combo))

    backtrack(dice_counts, (), 0, 0)
    return set(results) # Убираем дубликаты

In [7]:
class Player():
    def __init__(self, w=[random.random() for _ in range(16)]) -> None:
        self.w1 = w[0]
        self.w2 = w[1]
        self.w3 = w[2]
        self.w4 = w[3]
        self.w5 = w[4]
        self.w6 = w[5]
        self.w7 = w[6]
        self.w8 = w[7]
        self.w9 = w[8]
        self.w10 = w[9]
        self.w11 = w[10]
        self.w12 = w[11]
        self.w13 = w[12]
        self.w14 = w[13]
        self.w15 = w[14]
        self.w16 = w[15]

        self.W = w

        self.path = []

    def get_w(self):
        return self.W

    def make_decision(self, total, you_have, opponent_have, dice_count, round_have, comb_list):
        choose = []
        choose.append(("skip", self.w6 * (self.w1*total + self.w2*you_have + self.w3*opponent_have + self.w4*dice_count + self.w5*round_have + sum([self.w7*s + self.w8*d for s,d in comb_list]))))
        for s, d in comb_list:
            choose.append(((s, d), self.w16 * (self.w9*total + self.w10*you_have + self.w11*opponent_have + self.w12*dice_count + self.w13*round_have + self.w14*s + self.w15*d)))

        return max(choose, key=lambda x: x[1])


In [8]:
def turn(player: Player, dice_count, total, player_have, opponent_have, round_have):

    roll = [random.randint(1, 6) for _ in range(dice_count)]

    all_found = find_scores(Counter(roll))
    #print(f"all_found: {all_found}")
    comb_list = [(s, 6-d) for _,s,d in all_found]

    player.path.append([total, player_have, opponent_have, dice_count, round_have, comb_list])

    if (len(comb_list) == 0):
        player.path[-1].append("lose")
        return ("lose",)

    decision = player.make_decision(total, player_have, opponent_have, dice_count, round_have, comb_list)

    if (decision[0] == 'skip'):
        player.path[-1].append("skip")
        return "skip", max(comb_list, key=lambda x: x[0])[0]
    else:
        player.path[-1].append(decision[0])
        return decision[0][0], decision[0][1]

def round(player: Player, total, player_have, opponent_have, dice_count, round_have):
    #print(f"\nstart with d: {dice_count} and r: {round_have} ")

    trn = turn(player, dice_count, total, player_have, opponent_have, round_have)
    #print(f"turn decision: {trn}")

    if trn[0] == "lose":
        return 0
    if trn[0] == "skip":
        return trn[1]
    return round(player, total, player_have, opponent_have, dice_count - trn[1], round_have + trn[0])



def game(player1: Player, player2, total, max_rounds=100):
    #print(f"player1_w: {player1.get_w()}")

    player1_have = 0
    player2_have = 0
    rounds = 0

    player1_round_scores = []
    player2_round_scores = []

    while (player1_have < total or player2_have < total):
        if rounds >= max_rounds:
            return (True, False, player1_round_scores, player2_round_scores) if player1_have > player2_have else (False, True, player1_round_scores, player2_round_scores)
        #print("\n------------------------------")
        #print("player1 round")
        player1_round_score = round(player1, total, player1_have, player2_have, 6, 0)
        player1_round_scores.append(player1_round_score)
        player1_have += player1_round_score
        if (player1_have >= total):
            return True, False, player1_round_scores, player2_round_scores
        #print("\n------------------------------")
        #print("player2 round")
        player2_round_score = round(player2, total, player2_have, player1_have, 6, 0)
        player2_round_scores.append(player2_round_score)
        player2_have += player2_round_score
        if (player2_have >= total):
            return False, True, player1_round_scores, player2_round_scores
        rounds += 1

In [65]:
def GenAlg(total, genCount, how_many_to_select, w_of_scrores = 1, init_distribution_of_w=[random.random() for _ in range(16)]):

    number_of_individuals = how_many_to_select * (how_many_to_select + 1)

    #Gen = GenOriginal.copy()

    Gen = [[Player(init_distribution_of_w), 0, []] for _ in range(number_of_individuals)]
    n = number_of_individuals

    for gen_i in range(genCount):
        print("\n-------------------")
        print(f"gen: {gen_i+1}")
        for i in range(n-1):
            for j in range(i+1, n):
                #print(f"game between: {i} and {j}")
                result = game(Gen[i][0], Gen[j][0], total)
                Gen[i][1] += result[0]
                Gen[j][1] += result[1]

                Gen[i][2] += result[2]
                Gen[j][2] += result[3]


        Gen.sort(key=lambda x: x[1]/((n**2-n)/2) + sum(np.array(x[2])**w_of_scrores)/(((n**2-n)/2)*8000**w_of_scrores))
        print(f"Gen{gen_i+1}: {[[np.round(gen[1]/((n**2-n)/2), 6)] + [np.round(sum(np.array(gen[2])**w_of_scrores)/(((n**2-n)/2)*8000**w_of_scrores), 6)] + [np.round(gen[1]/((n**2-n)/2)+sum(np.array(gen[2])**w_of_scrores)/(((n**2-n)/2)*8000**w_of_scrores), 6)] for gen in Gen]}")

        if (gen_i == genCount-1):
            break

        winners = [player[0] for player in Gen[-how_many_to_select:]]

        for winner in winners:
            winner.path = []

        winners_w = [np.array(winner.get_w()) for winner in winners]

        K = [np.random.rand(len(winner_w.tolist())) for winner_w in winners_w]

        C_w = []
        for i in range(how_many_to_select-1):
            for j in range(i+1, how_many_to_select):
                C_w.append((1 - K[i]) * winners_w[i] + K[i] * winners_w[j])

        Gen = [[winner, 0, []] for winner in winners]
        for c_w in C_w:
            Gen.append([Player(c_w), 0, []])

        for i in range(n):
            Gen.append([Player(np.array(Gen[i][0].get_w()) + np.random.normal(size=16)), 0, []])

    return Gen


In [66]:
def individual_stats(individ: Player):
    df = pd.DataFrame(individ.path)
    print(df[6].apply(lambda x: 'move' if isinstance(x, tuple) else x).value_counts())
    #print(df[6].value_counts())

def create_brood(total, genCount, how_many_to_select, w_of_scrores = 1, init_distribution_of_w=[random.random() for _ in range(16)]):
    brood = GenAlg(total, genCount, how_many_to_select, w_of_scrores, init_distribution_of_w)
    print(len(brood))
    individual_stats(brood[-1][0])
    return brood

In [67]:
brood1 = create_brood(4000, 3, 5, w_of_scrores = 1.5)


-------------------
gen: 1
Gen1: [[0.022989, 0.007698, 0.030686], [0.022989, 0.007759, 0.030747], [0.022989, 0.008408, 0.031396], [0.025287, 0.007492, 0.032779], [0.025287, 0.008138, 0.033426], [0.025287, 0.008691, 0.033978], [0.027586, 0.007761, 0.035347], [0.029885, 0.007789, 0.037674], [0.029885, 0.008237, 0.038122], [0.029885, 0.008453, 0.038338], [0.032184, 0.00779, 0.039974], [0.032184, 0.007834, 0.040018], [0.032184, 0.008269, 0.040453], [0.032184, 0.008518, 0.040702], [0.034483, 0.008298, 0.042781], [0.034483, 0.008593, 0.043076], [0.034483, 0.008916, 0.043399], [0.034483, 0.00944, 0.043923], [0.036782, 0.008281, 0.045063], [0.036782, 0.008577, 0.045359], [0.036782, 0.008856, 0.045637], [0.03908, 0.008789, 0.04787], [0.03908, 0.009005, 0.048085], [0.03908, 0.009126, 0.048206], [0.03908, 0.009262, 0.048342], [0.03908, 0.009319, 0.048399], [0.041379, 0.008963, 0.050342], [0.041379, 0.009563, 0.050942], [0.041379, 0.009903, 0.051282], [0.041379, 0.010295, 0.051675]]

------------

In [68]:
brood2 = create_brood(4000, 3, 5, w_of_scrores = 2.5)


-------------------
gen: 1
Gen1: [[0.018391, 0.001073, 0.019464], [0.022989, 0.00075, 0.023739], [0.022989, 0.000793, 0.023782], [0.025287, 0.000864, 0.026151], [0.027586, 0.000654, 0.02824], [0.027586, 0.000801, 0.028388], [0.027586, 0.001136, 0.028722], [0.027586, 0.001407, 0.028993], [0.029885, 0.000715, 0.0306], [0.029885, 0.000781, 0.030666], [0.029885, 0.000811, 0.030696], [0.032184, 0.000727, 0.032911], [0.032184, 0.000868, 0.033051], [0.032184, 0.000886, 0.03307], [0.032184, 0.000968, 0.033152], [0.032184, 0.000979, 0.033163], [0.032184, 0.001393, 0.033577], [0.034483, 0.001053, 0.035536], [0.034483, 0.001585, 0.036068], [0.036782, 0.000915, 0.037697], [0.036782, 0.000998, 0.037779], [0.036782, 0.001079, 0.037861], [0.036782, 0.001159, 0.037941], [0.036782, 0.001763, 0.038545], [0.03908, 0.001034, 0.040114], [0.041379, 0.001382, 0.042761], [0.043678, 0.001006, 0.044684], [0.043678, 0.001444, 0.045123], [0.048276, 0.001212, 0.049488], [0.048276, 0.0022, 0.050476]]

------------

In [71]:
def save_w(player):
    import json

    with open("w.json", "w") as f:
        json.dump(player.get_w(), f)

In [ ]:
#save_w(brood2[-1][0])